# Reproduce TruthfulQA baselines (ported from `src/odesteer/`)

This notebook reproduces the TruthfulQA results table for the 11 ported baselines (RepE, CAA, ITI, MiMiC, LinAcT, SphericalSteer, COBRAS, ODESteer, RFFODESteer, StepODESteer, RFFStepODESteer) using AISteer360's framework.

Pipeline matches the source repo:
- 2-fold cross-validation on `pos/neg` activations precomputed at the target layer.
- System prompt: literal/myth-resistant TQA wording.
- Sampling: `temperature=0.7, top_p=0.9, repetition_penalty=1.1, max_new_tokens=50, do_sample=True`.
- Hook: replace last-token hidden state of the target layer with the `Steer.steer(h, T=...)` output.
- Judges: AllenAI `truthfulqa-truth-judge-llama2-7B` and `truthfulqa-info-judge-llama2-7B`.
- Quality: GPT2-XL perplexity + Dist-1/2/3.

Set `MODEL`, `DATA_MODEL_NAME`, `LAYER`, `METHODS`, etc. below.

In [ ]:
import gc
import json
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from lightning import seed_everything

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.evaluation.use_cases.truthful_qa import TruthfulQA
from aisteer360.evaluation.use_cases.truthful_qa.data import load_tqa_gen_data
from aisteer360.evaluation.metrics.custom.truthful_qa.allenai_judges import (
    AllenAITruthfulness, AllenAIInformativeness,
)
from aisteer360.evaluation.metrics.custom.truthful_qa.quality import (
    DistinctN, GPT2XLPerplexity,
)

In [ ]:
MODEL = 'meta-llama/Llama-3.1-8B'
DATA_MODEL_NAME = 'Llama3.1-8B-Base'
LAYER = 13
SEED = 42
BATCH_SIZE = 4
DTYPE = torch.float32
DATA_DIR = None
NUM_SAMPLES = -1
RESULTS_DIR = Path('results') / DATA_MODEL_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    ('repe', 1.0),
    ('caa', 1.0),
    ('iti', 1.0),
    ('mimic', 1.0),
    ('lin_act', 1.0),
    ('sphere_steer', 1.0),
    ('cobras', 0.5),
    ('ode_steer', 1.0),
    ('rff_ode_steer', 1.0),
    ('step_ode_steer', 1.0),
    ('rff_step_ode_steer', 1.0),
]

## Generate responses for each method (heavy)

In [ ]:
use_case = TruthfulQA(
    model_name=DATA_MODEL_NAME,
    layer_idx=LAYER,
    evaluation_metrics=[],
    data_dir=DATA_DIR,
    num_samples=NUM_SAMPLES,
)

for method_name, T in METHODS:
    raw_path = RESULTS_DIR / f'l{LAYER}-{method_name}-T{T}-seed{SEED}.jsonl'
    if raw_path.exists():
        print(f'[{method_name}] reusing {raw_path}')
        continue
    seed_everything(SEED)
    mod = importlib.import_module(f'aisteer360.algorithms.state_control.{method_name}')
    control = mod.STEERING_METHOD['control'](layer_id=LAYER, T=T)
    pipeline = SteeringPipeline(
        model_name_or_path=MODEL,
        controls=[control],
        device_map='auto',
        hf_model_kwargs={'torch_dtype': DTYPE},
    )
    pipeline.tokenizer.padding_side = 'left'
    if pipeline.tokenizer.pad_token is None:
        pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
    pos1, neg1 = load_tqa_gen_data(DATA_MODEL_NAME, LAYER, 1, DATA_DIR)
    pipeline.steer(pos_activations=pos1, neg_activations=neg1)

    gens = use_case.generate(
        model_or_pipeline=pipeline,
        tokenizer=pipeline.tokenizer,
        batch_size=BATCH_SIZE,
    )
    with open(raw_path, 'w') as f:
        for g in gens:
            f.write(json.dumps({
                'prompt': g['question'],
                'output': g['response'],
                'split': g['split'],
                'correct_answers': g['correct_answers'],
                'incorrect_answers': g['incorrect_answers'],
                'generator': f'{DATA_MODEL_NAME}-{method_name}',
                'T': T,
            }) + '\n')
    print(f'[{method_name}] wrote {len(gens)} rows')
    del pipeline
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Evaluate (judges + quality)

In [ ]:
truth_metric = AllenAITruthfulness()
info_metric = AllenAIInformativeness()
ppl_metric = GPT2XLPerplexity()
d1, d2, d3 = DistinctN(1), DistinctN(2), DistinctN(3)

rows = []
for method_name, T in METHODS:
    raw_path = RESULTS_DIR / f'l{LAYER}-{method_name}-T{T}-seed{SEED}.jsonl'
    if not raw_path.exists():
        print(f'skip {method_name}: no raw file at {raw_path}')
        continue
    with open(raw_path) as f:
        gens = [json.loads(line) for line in f]
    gens_for_metric = [
        {'response': g['output'], 'question': g['prompt'],
         'correct_answers': g.get('correct_answers', []),
         'incorrect_answers': g.get('incorrect_answers', [])}
        for g in gens
    ]
    truth = truth_metric(responses=gens_for_metric)
    info = info_metric(responses=gens_for_metric)
    ppl = ppl_metric(responses=gens_for_metric)
    s1 = d1(responses=gens_for_metric)
    s2 = d2(responses=gens_for_metric)
    s3 = d3(responses=gens_for_metric)
    ts = np.asarray(truth['scores']); is_ = np.asarray(info['scores'])
    rows.append({
        'Model': f'{DATA_MODEL_NAME}-l{LAYER}',
        'Steering Method': f'{method_name}-T{T}',
        'True * Info': float((ts * is_).mean()) if ts.size else 0.0,
        'Truthfulness': truth['truthfulness_rate'],
        'Informativeness': info['informativeness_rate'],
        'Perplexity': ppl['perplexity'],
        'Dist-1': s1['dist_1'],
        'Dist-2': s2['dist_2'],
        'Dist-3': s3['dist_3'],
    })

df = pd.DataFrame(rows)
df

In [ ]:
summary_path = RESULTS_DIR / f'l{LAYER}-TruthfulQA-seed{SEED}.csv'
df.to_csv(summary_path, index=False)
print(f'wrote {summary_path}')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df['Steering Method'], df['True * Info'])
ax.set_ylabel('True \u00d7 Info')
ax.set_xticklabels(df['Steering Method'], rotation=45, ha='right')
ax.set_title(f'TruthfulQA True * Info by method ({DATA_MODEL_NAME} layer {LAYER})')
plt.tight_layout()
plt.show()